In [ ]:
#使用pytorch实现mlp模型，使用MNIST数据集训练
from collections import defaultdict

import dnnlpy
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as utils
import torchvision.datasets as datasets
import torchvision.transforms.v2 as v2
from torch import Tensor

In [ ]:
#准备MNIST数据集
#数据存放路径root
root = dnnlpy.get_data_root()

#transform将mnist数据集转换成torch的tensor形式：(通道数, 高, 宽)，即(1，28，28)。并且把像素值缩放到[0,1]
transform = v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])

#训练&测试数据
train_ds = datasets.MNIST(root, train=True, download=True, transform=transform)
test_ds = datasets.MNIST(root, train=False, download=True, transform=transform)

#DataLoader：Torch的DataLoader 会自动帮我们完成打乱数据、分batch和迭代（迭代就是总样本数1280，每次取128个，迭代10次取完）
train_dl = utils.DataLoader(train_ds, batch_size=128, shuffle=True)
test_dl = utils.DataLoader(test_ds, batch_size=256, shuffle=False)

In [ ]:
#定义训练设备device
device=dnnlpy.get_default_device()
print('Using device:',device)

In [ ]:
#创建mlp模型、损失函数、优化器
model=MLP(input_dim=784,hidden_dim=256,classes_num=10).to(device)
loss_fn=nn.CrossEntropyLoss()
optimizer=optim.SGD(model.parameters(),lr=0.1)
history=defaultdict(list)

In [ ]:
#训练
num_epochs=10
for epoch in range(1,num_epochs+1):
    model.train() #模型进入训练模式，开启dropout和BatchNorm

    train_loss=0.0
    correct_samples=0

    #从train_dl中每次迭代batch_size大小的数据来训练，直到整个train_ds被迭代完
    for X,y in train_dl:
        X=X.to(device) #将数据读取到gpu上运行
        y=y.to(device)

        logits=model(X)
        loss=loss_fn(logits,y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss+=loss.item()
        correct_samples+=(logits.argmax(dim=1)==y).sum().item()

    #记录本次epoch训练过程的平均损失、平均准确率
    avg_train_loss=train_loss/len(train_dl)
    avg_train_acc=correct_samples/len(train_ds)
    history['loss'].append(avg_train_loss)
    history['acc'].append(avg_train_acc)